# 05 – SVM tuyến tính: ablation đặc trưng, tinh chỉnh C, kiểm chứng Group split

**Người thực hiện:** danh (Huỳnh Quốc Danh)  
**Mục tiêu:** Trả lời RQ1 (đặc trưng nào giúp) và chọn mô hình cuối cùng cho hệ thống.

**Quy tắc:** chạy TỪNG ô một, đổi tham số rồi chạy lại; mỗi ngày làm xong thì `git add` + `git commit -m "<mô tả đúng việc đã làm>"` + `git push`. Cuối buổi xuất HTML: `python ../scripts/export_notebook_html.py <notebook> danh`.

**Các bước:**
1. Ablation: word → char → hybrid → full (cùng C).
2. Thử char_ngram (2,4), (3,5), (3,6).
3. Quét C ∈ {0.03, 0.1, 0.3, 1, 3} có CV.
4. class_weight='balanced' có giúp 3 lớp tủ lạnh/tủ đông?
5. Chạy lại cấu hình tốt nhất với split='group'.
6. GridSearchCV nhỏ (tùy chọn) và kết luận mô hình chọn.

In [ ]:
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.append('../src')
from run_experiment import run
from figlog import save_fig
from exp_logger import log_generic
from common import load_data, clean_title


: 

## 1. Ablation đặc trưng

In [7]:
res, _ = run(member='danh', model='svc', features='word', params={'C': 1.0}, exp_name='abl_word', show_report=False)

[danh_20261113_222803] svc/word {'C': 1.0} split=strat
   train_time_s                 1.9970
   test_accuracy                0.9669
   test_precision_macro         0.9685
   test_recall_macro            0.9674
   test_f1_macro                0.9679
   test_f1_weighted             0.9669
   f1_Fridges                   0.9234
   f1_Freezers                  0.9530
   f1_Fridge_Freezers           0.9495
→ Đã ghi vào experiments/danh/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính:SVM (LinearSVC, C=1) chỉ dùng word TF-IDF (1,2): Macro-F1 0.9679, Accuracy 0.9669; F1 Fridges 0.9234, Freezers 0.9530, Fridge Freezers 0.9495 (run_id danh_20261113_164215). Chạy lại ra đúng bằng ngày 12/11 (danh_20261112_124307) → kết quả tái lập được.
> - Nhận xét: đây là mốc so sánh. Ba lớp tủ lạnh/tủ đông yếu nhất (đều dưới 0.96), Fridges thấp nhất, vì nhiều tiêu đề chỉ có hãng + mã model mà word n-gram không đọc được. Thêm char n-gram (hybrid 2,4) tăng lên 0.9924 (+2,45 điểm); thêm hình thái + merchant (full) chỉ được 0.9920, không hơn hybrid.
> - Lần sau thử: dùng hybrid làm đặc trưng chính, so phạm vi char (2,4)/(3,5)/(2,5)/(3,6) bằng 5-fold CV.

Đổi `features` thành `'char'`, `'hybrid'`, `'full'` và chạy lại – mỗi lần ghi nhận xét.

## 2. Phạm vi n-gram ký tự

In [8]:
res, _ = run(member='danh', model='svc', features='hybrid', params={'C': 1.0}, char_ngram=(2, 4), exp_name='char_24', show_report=False)

[danh_20261113_222855] svc/hybrid {'C': 1.0} split=strat
   train_time_s                 6.3731
   test_accuracy                0.9929
   test_precision_macro         0.9927
   test_recall_macro            0.9921
   test_f1_macro                0.9924
   test_f1_weighted             0.9929
   f1_Fridges                   0.9818
   f1_Freezers                  0.9739
   f1_Fridge_Freezers           0.9918
→ Đã ghi vào experiments/danh/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: SVM (C=1) với hybrid TF-IDF = word (1,2) + char_wb (2,4): Macro-F1 0.9924, Accuracy 0.9929; F1 Fridges 0.9818, Freezers 0.9739, Fridge Freezers 0.9918; train 6.6 s.
> - Nhận xét:  so với chỉ dùng word (0.9679), thêm n-gram ký tự tăng +2,45 điểm; Fridges tăng mạnh nhất (0.923 → 0.982). Lý do: char n-gram đọc được tiền tố mã model (vd "kif", "ks") và từ tiếng Đức mất dấu mà word n-gram bỏ sót. Các phạm vi khác: (2,5) 0.9924 nhưng chậm hơn, (3,5) 0.9917, (3,6) 0.9910. CV xác nhận (2,4) = 0.99066 ± 0.00095 cao hơn (3,5) = 0.98890 ± 0.00080, chênh lớn hơn std nên tốt hơn thật.
> - Lần sau thử: giữ char (2,4), quét C = 1, 3, 10 có CV.

## 3. Quét C (có 5-fold CV)

In [9]:
res, _ = run(member='danh', model='svc', features='full', params={'C': 0.3}, cv=True, exp_name='C_0.3', show_report=False)

[danh_20261113_223018] svc/full {'C': 0.3} split=strat
   cv_f1_macro_mean             0.9866
   cv_f1_macro_std              0.0010
   cv_time_s                    60.5743
   train_time_s                 14.8170
   test_accuracy                0.9904
   test_precision_macro         0.9905
   test_recall_macro            0.9895
   test_f1_macro                0.9900
   test_f1_weighted             0.9904
   f1_Fridges                   0.9709
   f1_Freezers                  0.9748
   f1_Fridge_Freezers           0.9891
→ Đã ghi vào experiments/danh/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: SVM full (hybrid char 3,5 + hình thái + merchant), C=0.3, 5-fold CV: CV Macro-F1 0.9866 ± 0.0010; Test Macro-F1 0.9900, Accuracy 0.9904; F1 Fridges 0.9709, Freezers 0.9748 (run_id danh_20261113_164354, trùng khớp ngày 12/11). Quét thêm bằng terminal với hybrid char (2,4): C=1 → CV 0.99066 ± 0.00095; C=3 → CV 0.99092 ± 0.00131, Test 0.9929, Freezers 0.9762; C=10 → CV 0.99049 ± 0.00161
> - Nhận xét: C nhỏ (0.3) phạt quá mạnh nên mô hình bị đơn giản hóa, F1 thấp hơn C=1 khoảng 0.2 điểm, Fridges giảm rõ nhất. C=1 và C=3 gần như ngang nhau (CV chênh 0.0003, nhỏ hơn std). C=10 bắt đầu giảm và std tăng, tức kém ổn định hơn.
> - Lần sau thử: chọn C=3 với hybrid char (2,4); chạy cấu hình này với split='group' và lưu làm mô hình cuối.

## 4. class_weight

In [10]:
res, _ = run(member='danh', model='svc', features='full', params={'C': 1.0, 'class_weight': 'balanced'}, exp_name='balanced')

[danh_20261113_223056] svc/full {'C': 1.0, 'class_weight': 'balanced'} split=strat
   train_time_s                 17.4056
   test_accuracy                0.9921
   test_precision_macro         0.9917
   test_recall_macro            0.9912
   test_f1_macro                0.9914
   test_f1_weighted             0.9921
   f1_Fridges                   0.9791
   f1_Freezers                  0.9727
   f1_Fridge_Freezers           0.9918
                  precision    recall  f1-score   support

            CPUs     0.9987    0.9987    0.9987       773
 Digital Cameras     0.9944    0.9944    0.9944       540
     Dishwashers     0.9913    0.9942    0.9927       685
        Freezers     0.9772    0.9683    0.9727       442
 Fridge Freezers     0.9918    0.9918    0.9918      1100
         Fridges     0.9804    0.9777    0.9791       717
      Microwaves     0.9979    0.9979    0.9979       468
   Mobile Phones     0.9951    1.0000    0.9976       816
             TVs     0.9972    0.9944    0

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: SVM full, C=1, class_weight='balanced': Macro-F1 0.9914, Accuracy 0.9921; F1 Fridges 0.9791, Freezers 0.9727, Fridge Freezers 0.9918 (run_id danh_20261113_164414, trùng khớp ngày 12/11). So với không cân bằng (full C=1): Macro-F1 0.9920, Freezers 0.9749.
> - Nhận xét: cân bằng trọng số lớp không giúp, Macro-F1 giảm 0.0006 và Freezers còn giảm nhẹ. Bảng phân loại cho thấy Freezers có recall thấp nhất (0.9683, tức 14/442 tủ đông bị đoán sang lớp khác). Freezers là lớp ít nhất (442 mẫu Test) nhưng mức lệch lớp nhẹ (~2,5 lần), nên lỗi chủ yếu do trùng từ vựng với Fridges/Fridge Freezers chứ không phải do thiếu mẫu. Các lớp điện tử (CPUs, Mobile Phones, Microwaves) gần như tuyệt đối (F1 ≥ 0.997).
> - Lần sau thử: bỏ class_weight; muốn cải thiện Freezers thì cần đặc trưng phân biệt tốt hơn.

## 5. Group split theo Cluster ID
Câu hỏi phải trả lời được: *vì sao F1 giảm khi chia theo sản phẩm? giảm bao nhiêu là chấp nhận được?*

In [11]:
res, _ = run(member='danh', model='svc', features='full', params={'C': 1.0}, split='group', exp_name='group_best', show_report=False)

[danh_20261113_223116] svc/full {'C': 1.0} split=group
   train_time_s                 17.9700
   test_accuracy                0.9871
   test_precision_macro         0.9877
   test_recall_macro            0.9860
   test_f1_macro                0.9868
   test_f1_weighted             0.9871
   f1_Fridges                   0.9614
   f1_Freezers                  0.9761
   f1_Fridge_Freezers           0.9804
→ Đã ghi vào experiments/danh/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: SVM full, C=1, chia GroupShuffleSplit theo Cluster ID (cùng một sản phẩm thực không nằm ở cả Train và Test): Macro-F1 0.9868, Accuracy 0.9871; F1 Fridges 0.9614, Freezers 0.9761, Fridge Freezers 0.9804; tập Test 7.154 mẫu (run_id danh_20261113_164436, trùng khớp ngày 12/11). So với chia ngẫu nhiên Stratified: 0.9920.
> - Nhận xét: F1 giảm 0.52 điểm khi tách hẳn sản phẩm. Lý do: khi chia ngẫu nhiên, nhiều offer trong Test có "anh em" cùng sản phẩm (tiêu đề gần giống) trong Train, nên kết quả hơi lạc quan. Fridges giảm nhiều nhất (0.9791 → 0.9614), cho thấy tủ lạnh phụ thuộc nhiều vào việc đã gặp đúng mẫu đó. Dù vậy, với sản phẩm hoàn toàn mới mô hình vẫn đạt ~0.987, tức là học được dấu hiệu chung của từng ngành chứ không chỉ học thuộc tiêu đề. Giữa kỳ, mô hình chỉ dùng word giảm 1,2–2,5 điểm; char n-gram giúp mô hình tổng quát tốt hơn.
> - Lần sau thử: chạy cấu hình cuối (hybrid char 2,4, C=3) với split='group' để báo cáo kết quả trên sản phẩm mới; khi tinh chỉnh cuối kỳ nên dùng GroupKFold.

## 6. Bảng so sánh & hình

In [12]:
log = pd.read_csv('../experiments/danh/experiment_log.csv')
log[['run_id','ten_thi_nghiem','features','params','split','test_f1_macro','f1_Fridges','f1_Freezers','nhan_xet']]

,run_id,ten_thi_nghiem,features,params,split,test_f1_macro,f1_Fridges,f1_Freezers,nhan_xet
0,danh_20261112_124307,abl_word,word,"{""C"": 1.0}",strat,0.967909,0.923398,0.953036,"F1 0.9679, khớp giữa kỳ. Ba lớp tủ lạnh/tủ đôn..."
1,danh_20261112_124329,char_24,hybrid,"{""C"": 1.0}",strat,0.992425,0.981844,0.973893,"F1 0.9924 (+2,45 điểm), Fridges tăng lên 0.982..."
2,danh_20261112_131842,C_0.3,full,"{""C"": 0.3}",strat,0.990015,0.970914,0.974828,"F1 0.9900, thấp hơn C=1 (0.9920). CV 0.9866 ± ..."
3,danh_20261112_131859,balanced,full,"{""C"": 1.0, ""class_weight"": ""balanced""}",strat,0.991449,0.979050,0.972727,"F1 0.9914, thấp hơn khi không cân bằng. Freeze..."
4,danh_20261112_131917,group_best,full,"{""C"": 1.0}",group,0.986821,0.961420,0.976067,"F1 0.9868, giảm 0,5 điểm. Fridges giảm nhiều n..."
5,danh_20261112_133323,abl_full,full,"{""C"": 1.0}",strat,0.991971,0.981158,0.974943,"F1 0.9920, khớp E06. Freezers thấp nhất (0.975..."
6,danh_20261113_164215,abl_word,word,"{""C"": 1.0}",strat,0.967909,0.923398,0.953036,NaN
7,danh_20261113_164224,char_24,hybrid,"{""C"": 1.0}",strat,0.992425,0.981844,0.973893,NaN
8,danh_20261113_164354,C_0.3,full,"{""C"": 0.3}",strat,0.990015,0.970914,0.974828,NaN
9,danh_20261113_164414,balanced,full,"{""C"": 1.0, ""class_weight"": ""balanced""}",strat,0.991449,0.979050,0.972727,NaN


## 7. Mô hình chọn và lý do 
- Đặc trưng:  Hybrid TF-IDF = word n-gram (1,2) + char_wb n-gram (2,4), sublinear_tf, min_df=2. Vì: thêm n-gram ký tự tăng Macro-F1 từ 0.9679 lên 0.9924 (+2,45 điểm; danh_20261113_164215 → _164224), do ~1/5 tiêu đề chỉ có hãng + mã model hoặc tiếng Đức mất dấu, word n-gram không đọc được còn char n-gram nắm được tiền tố mã model. Phạm vi (2,4) tốt hơn (3,5) thật sự trên 5-fold CV: 0.99066 ± 0.00095 so với 0.98890 ± 0.00080, chênh 0.0018 > std (_170717, _170850); (3,6) kém hơn (0.9910), (2,5) ngang nhưng chậm hơn. Không dùng thêm đặc trưng hình thái + merchant vì không cải thiện (full 0.9920 ≤ hybrid 0.9924). 
- Mô hình: LinearSVC (SVM tuyến tính). Vì: tốt nhất trong các mô hình đã thử (NB 0.9663, Logistic 0.9656, Decision Tree 0.9408, XGBoost 0.9771, Random Forest 0.9733 – theo nhật ký nhóm và giữa kỳ), phù hợp dữ liệu văn bản thưa nhiều chiều, huấn luyện chỉ vài giây. class_weight='balanced' không giúp (0.9914 < 0.9920; _164414) nên giữ mặc định.
- Tham số:   Kiểm chứng trên sản phẩm mới (Group split theo Cluster ID): Macro-F1 0.9875 (danh_20261113_231233), giảm 0.54 điểm so với chia ngẫu nhiên – mô hình vẫn tổng quát tốt.